<div dir="rtl">
<h1>از فرمول Affine تا Parameter ثبت‌شده</h1>
<p>درس 22 از 76 · یک Layer خطی و ثبت Parameterها · <code dir="ltr">18-module</code></p>
<p><a target="_self" href="http://127.0.0.1:8000/part-03/chapter-03/18-module.html">📖 بازگشت به همین درس</a></p>
<p>محاسبهٔ Layer و ثبت وزن در Module را جدا بسازید و بررسی کنید.</p><p>پیش‌نیاز: 07-matmul،17-autograd و nn.Module در درس جاری.</p>
<p>این دفتر نیمهٔ عملی درس است. مثال‌ها آمادهٔ اجرا هستند؛ دو Cell با برچسب TODO را خودتان کامل کنید. پیام INCOMPLETE یعنی هنوز چیزی ننوشته‌اید، نه اینکه پاسخ درست است. جواب مرجع در این دفتر پنهان نشده است.</p>
<p>از بالا به پایین اجرا کنید. پس از تغییر هر تابع، Cell آن و سپس Cell آزمون را دوباره اجرا کنید. برای بررسی نهایی، از منوی <code>Kernel → Restart Kernel and Run All Cells</code> استفاده کنید.</p>
</div>

In [ ]:
from pathlib import Path
import os
import sys

project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "mini_gpt").is_dir() and (p / "book_src").is_dir()), None)
if project_root is None:
    raise RuntimeError("Extract the complete learning project; open this notebook inside it.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print("Python:", sys.executable)
print("Project:", project_root)

<div dir="rtl">
<h2>قبل از اجرا، پیش‌بینی کنید</h2>
<p>با W سه‌در‌دو، ورودی یک‌در‌دو و Bias سه‌تایی، خروجی چند عدد دارد؟ آیا هر Tensor دارای Gradient خودکار در parameters پیدا می‌شود؟</p>
</div>

<div dir="rtl"><p>پیش‌بینی من: …</p></div>

In [ ]:
import torch
from torch import nn
x = torch.tensor([[1., 2.]])
weight = torch.tensor([[0.5, -0.25], [1., 0.], [0., 1.]])
bias = torch.tensor([1., -2., 0.])
print("Input and weight shapes:", x.shape, weight.shape)

<div dir="rtl">
<h2>این بار شما کد بنویسید</h2>
<p>تابع affine(x,Weight,Bias) محاسبهٔ nn.Linear را با @، Transpose و جمع بنویسد. وزن به شکل (out,in) است؛ کپی یک nn.Linear تازه با وزن متفاوت پاسخ این تمرین نیست.</p>
</div>

In [ ]:
def affine(x, weight, bias):
    # TODO: use the supplied weight and bias
    return None

In [ ]:
def test_exercise():
    result = affine(x, weight, bias)
    if result is None:
        return False
    assert torch.equal(result, torch.tensor([[1., -1., 2.]]))
    assert tuple(affine(x.repeat(2, 1), weight, bias).shape) == (2, 3)
    assert torch.equal(affine(torch.zeros_like(x), weight, bias), bias[None])
    return True

exercise_complete = test_exercise()
print('PASS' if exercise_complete else 'INCOMPLETE: implement the TODO and rerun')

<div dir="rtl">
<h2>فقط یک عامل را تغییر دهید</h2>
<p>فقط Bias را صفر کنید؛ وزن و ورودی همان باشند. اختلاف خروجی‌ها باید برابر همان Bias باشد، نه وابسته به تعداد نمونه‌ها. این مقایسه از تابع آمادهٔ F.linear استفاده می‌کند؛ بازسازی محاسبه با @ همچنان کار شما در تمرین است.</p>
</div>

In [ ]:
from torch.nn import functional as F

with_bias = F.linear(x, weight, bias)
without_bias = F.linear(x, weight)
print("Difference:", with_bias-without_bias)
assert torch.equal(with_bias-without_bias, bias[None])

<div dir="rtl">
<h2>خرابی را پیدا کنید</h2>
<p>Tensor معمولی حتی با requires_grad=True به‌عنوان Parameter ثبت نمی‌شود. تابع registered_affine یک nn.Module با Weight و Bias ثبت‌شده و forward درست برگرداند؛ از وزن‌های ورودی کپی مستقل بگیرید.</p>
</div>

In [ ]:
broken = nn.Module()
broken.weight = weight.clone().requires_grad_()
print("Unregistered parameters:", list(broken.parameters()))
assert list(broken.parameters()) == []

<div dir="rtl">
<h2>اصلاح را خودتان بنویسید</h2>
<p>علت را توضیح دهید، سپس تابع زیر را کامل کنید. خطای عمدی بالا یک نمونهٔ آموزشی است؛ آزمون پایین باید اصلاح شما را بسنجد.</p>
</div>

In [ ]:
def registered_affine(weight, bias):
    # TODO: return a callable Module with registered parameters
    return None

In [ ]:
def test_repair():
    result = registered_affine(weight, bias)
    if result is None:
        return False
    assert sum(p.numel() for p in result.parameters()) == 9
    assert set(dict(result.named_parameters())) == {"weight", "bias"}
    assert torch.equal(result(x), torch.tensor([[1., -1., 2.]]))
    assert result.weight.data_ptr() != weight.data_ptr()
    result(x).sum().backward()
    assert result.weight.grad is not None and result.bias.grad is not None
    return True

repair_complete = test_repair()
print('PASS' if repair_complete else 'INCOMPLETE: implement the TODO and rerun')

<div dir="rtl">
<h2>در Mini-GPT کجا به کار می‌آید؟</h2>
<p>Head در mini_gpt/stages/v1.py از nn.Linear استفاده می‌کند. محاسبهٔ این Layer را ساختید و دیدید چرا ثبت Parameter، موضوعی جدا از درست‌بودن فرمول است.</p>
</div>

<div dir="rtl">
<h2>با زبان خودتان توضیح دهید</h2>
<p>کدام آزمون فرمول را بررسی کرد و کدام آزمون امکان پیدا‌کردن وزن‌ها توسط Optimizer را؟</p>
</div>
<div dir="rtl"><p>پیش‌بینی و مشاهدهٔ من: …</p><p>علت خرابی و اصلاح من: …</p></div>

<div dir="rtl"><p><a target="_self" href="http://127.0.0.1:8000/part-03/chapter-03/18-module.html">بازگشت به درس و ادامهٔ مسیر</a> · <a target="_self" href="http://127.0.0.1:8000/answers/18-module.html#lab-solution">فقط پس از تلاش: راه‌حل مرجع آزمایشگاه</a></p></div>